In [18]:
# Enhanced Label Studio Image Crop Merger with Proper Annotation Import
# Fixes annotation import issues and handles both keypoints and polygons

import os
import re
import json
import requests
from PIL import Image
import numpy as np
from pathlib import Path
from label_studio_sdk import Client
from urllib.parse import urlparse, unquote
import matplotlib.pyplot as plt
from collections import defaultdict
import io
import uuid
from datetime import datetime
from icecream import ic


class EnhancedCropMerger:
    def __init__(self, ls_url, api_token, project_id, local_files_root):
        self.ls_url = ls_url.rstrip('/')
        self.api_token = api_token
        self.project_id = project_id
        self.local_files_root = local_files_root
        self.client = Client(url=ls_url, api_key=api_token)
        self.project = self.client.get_project(project_id)
        
        # Set environment
        os.environ['LOCAL_FILES_DOCUMENT_ROOT'] = local_files_root
        
    def extract_crop_info(self, filename):
        """Extract original name and grid position from crop filename"""
        # Pattern: [uuid-uuid-uuid-]originalname_crop_row##_col##.ext
        crop_pattern = r'(.+?)_crop_row(\d+)_col(\d+)\.(.*?)$'
        match = re.match(crop_pattern, filename)
        
        if not match:
            return None
            
        base_name, row, col, ext = match.groups()
        
        # Check for UUID prefix
        uuid_pattern = r'^([a-f0-9]{8})-([a-f0-9]{8})-([a-f0-9]{8})-(.+)$'
        uuid_match = re.match(uuid_pattern, base_name)
        
        original_name = uuid_match.group(4) if uuid_match else base_name
        
        return {
            'original_name': original_name,
            'row': int(row),
            'col': int(col),
            'extension': ext,
            'has_uuid': bool(uuid_match)
        }
    
    def get_file_path(self, image_url):
        """Extract filename from Label Studio URL"""
        # ic(image_url)
        # ic(self.ls_url)
        # url = image_url.strip(self.ls_url)
        url = image_url
        ic(url)
        
        if url.startswith("/data/upload"):
            return os.path.basename(urlparse(url).path)
        elif url.startswith("/data/local-files") and "?d=" in url:
            path_part = unquote(url.split('?d=')[1])
            return os.path.basename(path_part)
        else:
            return os.path.basename(urlparse(url).path)
    
    def scan_for_crops(self):
        """Scan project for crop images and group by original"""
        print("🔍 Scanning project for crop images...")
        
        try:
            tasks = self.project.get_tasks()
        except Exception as e:
            print(f"❌ Error fetching tasks: {e}")
            return {}
        
        crop_groups = defaultdict(list)
        
        for task in tasks:
            data = task.get('data', {})
            image_url = data.get('image') or data.get('img')
            
            if not image_url:
                continue
                
            filename = self.get_file_path(image_url)
            ic(filename)
            crop_info = self.extract_crop_info(filename)
            
            if crop_info:
                crop_info.update({
                    'task_id': task['id'],
                    'image_url': image_url,
                    'task': task,
                    'filename': filename
                })
                crop_groups[crop_info['original_name']].append(crop_info)
        
        print(f"📊 Found {len(crop_groups)} original images with crops")
        return dict(crop_groups)
    
    def analyze_crop_grid(self, crops):
        """Analyze crop positions to determine grid size"""
        if not crops:
            return 0, 0, False
            
        rows = [c['row'] for c in crops]
        cols = [c['col'] for c in crops]
        
        grid_height = max(rows) + 1
        grid_width = max(cols) + 1
        expected_count = grid_height * grid_width
        
        positions = {(c['row'], c['col']) for c in crops}
        is_complete = len(positions) == expected_count
        
        return grid_height, grid_width, is_complete
    
    def download_image(self, image_url):
        """Download image from Label Studio"""
        try:
            if image_url.startswith('/'):
                full_url = f"{self.ls_url}{image_url}"
            else:
                full_url = image_url
                
            headers = {'Authorization': f'Token {self.api_token}'}
            response = requests.get(full_url, headers=headers)
            response.raise_for_status()
            
            return Image.open(io.BytesIO(response.content))
        except Exception as e:
            print(f"❌ Download failed: {e}")
            return None
    
    def merge_images(self, crops):
        """Merge crop images back into original"""
        grid_height, grid_width, is_complete = self.analyze_crop_grid(crops)
        
        if not is_complete:
            print(f"⚠️ Grid incomplete: {len(crops)}/{grid_height * grid_width} crops")
        
        # Download all crop images
        print(f"📥 Downloading {len(crops)} crop images...")
        crop_images = {}
        
        for i, crop in enumerate(crops):
            print(f"  📥 [{i+1}/{len(crops)}] {crop['filename']}")
            img = self.download_image(crop['image_url'])
            if img:
                crop_images[(crop['row'], crop['col'])] = img
                # Store dimensions for coordinate transformation
                crop['image_width'] = img.size[0]
                crop['image_height'] = img.size[1]
        
        if not crop_images:
            print("❌ No images downloaded successfully")
            return None, None
        
        # Get crop dimensions
        first_img = next(iter(crop_images.values()))
        crop_width, crop_height = first_img.size
        
        # Create merged image
        merged_width = grid_width * crop_width
        merged_height = grid_height * crop_height
        merged_img = Image.new('RGB', (merged_width, merged_height), 'white')
        
        print(f"🖼️ Creating merged image: {merged_width}x{merged_height}")
        
        # Place crops in grid
        for row in range(grid_height):
            for col in range(grid_width):
                if (row, col) in crop_images:
                    x = col * crop_width
                    y = row * crop_height
                    merged_img.paste(crop_images[(row, col)], (x, y))
        
        return merged_img, (crop_width, crop_height)
    
    def transform_coordinates(self, value, crop_info, crop_dims, merged_dims):
        """Transform coordinates from crop to merged image"""
        crop_width, crop_height = crop_dims
        merged_width, merged_height = merged_dims
        
        if value.get('x') is not None and value.get('y') is not None:
            # Keypoint coordinates
            crop_x_pct = value.get('x', 0)
            crop_y_pct = value.get('y', 0)
            
            # Convert to pixels within crop
            crop_x_px = (crop_x_pct / 100) * crop_width
            crop_y_px = (crop_y_pct / 100) * crop_height
            
            # Convert to pixels in merged image
            merged_x_px = (crop_info['col'] * crop_width) + crop_x_px
            merged_y_px = (crop_info['row'] * crop_height) + crop_y_px
            
            # Convert back to percentage
            merged_x_pct = (merged_x_px / merged_width) * 100
            merged_y_pct = (merged_y_px / merged_height) * 100
            
            return {
                **value,
                'x': merged_x_pct,
                'y': merged_y_pct
            }
        
        elif value.get('points') is not None:
            # Polygon coordinates
            points = value.get('points', [])
            transformed_points = []
            
            for point in points:
                if len(point) >= 2:
                    crop_x_pct, crop_y_pct = point[0], point[1]
                    
                    # Convert to pixels within crop
                    crop_x_px = (crop_x_pct / 100) * crop_width
                    crop_y_px = (crop_y_pct / 100) * crop_height
                    
                    # Convert to pixels in merged image
                    merged_x_px = (crop_info['col'] * crop_width) + crop_x_px
                    merged_y_px = (crop_info['row'] * crop_height) + crop_y_px
                    
                    # Convert back to percentage
                    merged_x_pct = (merged_x_px / merged_width) * 100
                    merged_y_pct = (merged_y_px / merged_height) * 100
                    
                    transformed_points.append([merged_x_pct, merged_y_pct])
            
            return {
                **value,
                'points': transformed_points
            }
        
        return value
    
    def merge_annotations(self, crops, crop_dims, merged_dims):
        """Merge annotations from crops back to original coordinates"""
        if not crops:
            return []
        
        merged_annotations = []
        
        for crop in crops:
            task = crop['task']
            annotations = task.get('annotations', [])
            
            for annotation in annotations:
                results = annotation.get('result', [])
                
                for result in results:
                    # Transform coordinates based on annotation type
                    if result.get('type') in ['keypointlabels', 'polygonlabels']:
                        value = result.get('value', {})
                        
                        # Transform coordinates
                        transformed_value = self.transform_coordinates(
                            value, crop, crop_dims, merged_dims
                        )
                        
                        # Create new annotation with transformed coordinates
                        new_result = {
                            **result,
                            'value': transformed_value,
                            'id': str(uuid.uuid4())[:10],  # New unique ID
                            'origin': 'manual'  # Mark as manual to ensure import
                        }
                        
                        # Update original dimensions
                        new_result['original_width'] = merged_dims[0]
                        new_result['original_height'] = merged_dims[1]
                        
                        merged_annotations.append(new_result)
        
        print(f"🎯 Merged {len(merged_annotations)} annotations")
        return merged_annotations
    
    def create_merged_task(self, original_name, merged_img, merged_annotations):
        """Create or update a task with merged image and annotations"""
        # Save merged image locally
        output_dir = Path("merged_images")
        output_dir.mkdir(exist_ok=True)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        merged_filename = f"{original_name}_merged_{timestamp}.jpg"
        local_path = output_dir / merged_filename
        merged_img.save(local_path, quality=95)

        print(f"💾 Saved locally: {local_path}")

        try:
            # Upload to Label Studio
            with open(local_path, 'rb') as f:
                files = {'file': (merged_filename, f, 'image/jpeg')}
                upload_url = f"{self.ls_url}/api/projects/{self.project_id}/import"
                headers = {'Authorization': f'Token {self.api_token}'}
                upload_response = requests.post(upload_url, files=files, headers=headers)

            if upload_response.status_code not in [200, 201]:
                print(f"❌ Upload failed: {upload_response.text}")
                return None

            # Search for existing task with the pattern <original_name>_merged in the image URL
            existing_tasks = self.project.get_tasks()
            task_id = None
            for task in existing_tasks:
                image_url = task.get("data", {}).get("image", "")
                if f"{original_name}_merged" in image_url:
                    print(f"🔁 Found existing task for {original_name}: {image_url}")
                    task_id = task["id"]
                    break

            if not task_id:
                # No existing task — create new
                task_data = {
                    'data': {
                        'image': f'/data/upload/{self.project_id}/{merged_filename}'
                    }
                }
                task_url = f"{self.ls_url}/api/projects/{self.project_id}/tasks/"
                task_response = requests.post(task_url, json=task_data, headers=headers)
                if task_response.status_code not in [200, 201]:
                    print(f"❌ Task creation failed: {task_response.text}")
                    return None
                task_id = task_response.json()["id"]
                print(f"🆕 Created new task ID: {task_id}")
            else:
                print(f"✅ Reusing existing task ID: {task_id}")

            # Add annotations if any
            if merged_annotations:
                annotation_data = {
                    'task': task_id,
                    'result': merged_annotations,
                    'was_cancelled': False,
                    'ground_truth': False,
                    'created_username': 'crop_merger',
                    'created_ago': '0 minutes'
                }

                annotation_url = f"{self.ls_url}/api/tasks/{task_id}/annotations/"
                annotation_response = requests.post(
                    annotation_url,
                    json=annotation_data,
                    headers=headers
                )

                if annotation_response.status_code not in [200, 201]:
                    print(f"⚠️ Annotation creation failed: {annotation_response.text}")
                    print("Task exists but annotations not imported")
                else:
                    print(f"✅ Annotations imported successfully")

            return {"id": task_id}

        except Exception as e:
            print(f"❌ Error creating or updating task: {e}")
            return None

    
    def process_image(self, original_name, crops):
        """Process a single image set"""
        print(f"\n🔄 Processing: {original_name}")
        print(f"📊 Crops: {len(crops)}")
        
        # Merge images
        merged_img, crop_dims = self.merge_images(crops)
        if not merged_img:
            print("❌ Image merge failed")
            return None
        
        merged_dims = merged_img.size
        
        # Merge annotations
        merged_annotations = self.merge_annotations(crops, crop_dims, merged_dims)
        
        # Create task
        new_task = self.create_merged_task(original_name, merged_img, merged_annotations)
        
        if new_task:
            print(f"🎉 Success! Task created with {len(merged_annotations)} annotations")
            return new_task
        
        return None
    
    def run_interactive_merge(self):
        """Run interactive merge process"""
        print("🚀 Starting Enhanced Crop Merger")
        print("=" * 50)
        
        # Scan for crops
        crop_data = self.scan_for_crops()
        
        if not crop_data:
            print("❌ No crop images found")
            return
        
        # Analyze crops
        print("\n📊 Crop Analysis:")
        print("-" * 30)
        
        complete_sets = []
        incomplete_sets = []
        
        for name, crops in crop_data.items():
            h, w, complete = self.analyze_crop_grid(crops)
            status = "✅ Complete" if complete else "⚠️ Incomplete"
            
            print(f"{status}: {name} ({len(crops)} crops, {h}x{w} grid)")
            
            if complete:
                complete_sets.append((name, crops))
            else:
                incomplete_sets.append((name, crops))
        
        print(f"\n📈 Summary: {len(complete_sets)} complete, {len(incomplete_sets)} incomplete")
        
        if not complete_sets:
            print("❌ No complete sets found for merging")
            return
        
        # Ask user what to do
        print("\n🎮 Options:")
        print("1. Merge all complete sets")
        print("2. Merge specific image")
        print("3. Show incomplete sets")
        
        choice = input("\nEnter choice (1-3): ").strip()
        
        if choice == "1":
            # Merge all complete sets
            print(f"\n🚀 Merging {len(complete_sets)} complete sets...")
            success_count = 0
            
            for i, (name, crops) in enumerate(complete_sets, 1):
                print(f"\n[{i}/{len(complete_sets)}] {name}")
                if self.process_image(name, crops):
                    success_count += 1
            
            print(f"\n🎉 Completed: {success_count}/{len(complete_sets)} successful")
        
        elif choice == "2":
            # Merge specific image
            print("\n📋 Available complete sets:")
            for i, (name, crops) in enumerate(complete_sets, 1):
                print(f"{i}. {name}")
            
            try:
                idx = int(input("\nEnter number: ")) - 1
                if 0 <= idx < len(complete_sets):
                    name, crops = complete_sets[idx]
                    self.process_image(name, crops)
                else:
                    print("❌ Invalid selection")
            except ValueError:
                print("❌ Invalid input")
        
        elif choice == "3":
            # Show incomplete sets
            print("\n⚠️ Incomplete sets:")
            for name, crops in incomplete_sets:
                h, w, _ = self.analyze_crop_grid(crops)
                missing = (h * w) - len(crops)
                print(f"  {name}: {len(crops)}/{h*w} crops (missing {missing})")
        
        print("\n✅ Merge process completed!")


# Configuration
LS_URL = 'http://129.97.250.147:8080/'
API_TOKEN = 'ebdc6fa5f2c3abcd502b55d5ccc1dc0e4ae9f68d'
PROJECT_ID = 108
LOCAL_FILES_ROOT = "/home/pc2041"

# Initialize and run
if __name__ == "__main__":
    merger = EnhancedCropMerger(LS_URL, API_TOKEN, PROJECT_ID, LOCAL_FILES_ROOT)
    merger.run_interactive_merge()

🚀 Starting Enhanced Crop Merger
🔍 Scanning project for crop images...


ic| url: '/data/local-files/?d=VIP_lab/labelstudio/data/2008/grants%20photos%201/BEL22to25exp/IMG_4199.JPG'
ic| filename: 'IMG_4199.JPG'
ic| url: '/data/local-files/?d=VIP_lab/labelstudio/data/2008/grants%20photos%201/BEL22to25exp/IMG_4219.JPG'
ic| filename: 'IMG_4219.JPG'
ic| url: '/data/local-files/?d=VIP_lab/labelstudio/data/2008/grants%20photos%201/BEL22to25exp/IMG_4235.JPG'
ic| filename: 'IMG_4235.JPG'
ic| url: '/data/local-files/?d=VIP_lab/labelstudio/data/2008/grants%20photos%201/BEL22to25exp/IMG_4242.JPG'
ic| filename: 'IMG_4242.JPG'
ic| url: '/data/local-files/?d=VIP_lab/labelstudio/data/2008/grants%20photos%201/BEL22to25exp/IMG_4243.JPG'
ic| filename: 'IMG_4243.JPG'
ic| url: '/data/local-files/?d=VIP_lab/labelstudio/data/2008/grants%20photos%201/BEL22to25exp/IMG_4245.JPG'
ic| filename: 'IMG_4245.JPG'
ic| url: '/data/local-files/?d=VIP_lab/labelstudio/data/2008/grants%20photos%201/BEL22to25exp/IMG_4246.JPG'
ic| filename: 'IMG_4246.JPG'
ic| url: '/data/local-files/?d=VIP_lab/la

📊 Found 2 original images with crops

📊 Crop Analysis:
------------------------------
✅ Complete: IMG_8710 (6 crops, 2x3 grid)
✅ Complete: IMG_8712 (6 crops, 2x3 grid)

📈 Summary: 2 complete, 0 incomplete

🎮 Options:
1. Merge all complete sets
2. Merge specific image
3. Show incomplete sets

🚀 Merging 2 complete sets...

[1/2] IMG_8710

🔄 Processing: IMG_8710
📊 Crops: 6
📥 Downloading 6 crop images...
  📥 [1/6] IMG_8710_crop_row00_col00.JPG
  📥 [2/6] IMG_8710_crop_row00_col01.JPG
  📥 [3/6] IMG_8710_crop_row00_col02.JPG
  📥 [4/6] IMG_8710_crop_row01_col00.JPG
  📥 [5/6] IMG_8710_crop_row01_col01.JPG
  📥 [6/6] IMG_8710_crop_row01_col02.JPG
🖼️ Creating merged image: 3072x2048
🎯 Merged 12941 annotations
💾 Saved locally: merged_images/IMG_8710_merged_20250710_115942.jpg
🔁 Found existing task for IMG_8710: /data/upload/108/b51aba2a-IMG_8710_merged_20250710_115942.jpg
✅ Reusing existing task ID: 73163
✅ Annotations imported successfully
🎉 Success! Task created with 12941 annotations

[2/2] IMG_